In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.models import Model
from skimage.feature import hog, daisy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [ ]:
# HOG + DAISY feature extractor
# ----------------------------
def extract_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # HOG
    hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys',
                       transform_sqrt=True, feature_vector=True)

    # DAISY
    daisy_desc = daisy(gray, step=8, radius=15, rings=2, histograms=6,
                       orientations=8, visualize=False)
    daisy_features = daisy_desc.flatten()

    # Match lengths
    min_len = min(len(hog_features), len(daisy_features))
    combined = np.hstack([hog_features[:min_len], daisy_features[:min_len]])
    return combined

# ----------------------------
# Load dataset from folders
# ----------------------------
def load_dataset(data_dir):
    X = []
    y = []

    for label in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, label)
        if not os.path.isdir(class_dir): continue

        for file in os.listdir(class_dir):
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(class_dir, file)
                img = cv2.imread(path)
                if img is None:
                    print(f"Failed to read {path}")
                    continue
                img = cv2.resize(img, (150, 150))

                try:
                    features = extract_features(img)
                    X.append(features)
                    y.append(label)
                except Exception as e:
                    print(f"Error processing {path}: {e}")

    return np.array(X), np.array(y)

In [ ]:
from sklearn.decomposition import PCA
import joblib

# X_train_features -> Feature vectors obtained after
# concatenating InceptionV3 + HOG + DAISY features

print("Original Feature Shape:", X_train_features.shape)

# Retain 95% of the variance
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_features)

print("Reduced Feature Shape:", X_train_pca.shape)

# Save the PCA model
joblib.dump(pca, "pca_model.pkl")

In [ ]:
if __name__ == '__main__':
    DATA_PATH = '/content/drive/MyDrive/Minor_Project_image_dataset/Final_Resized'  # Change this to your dataset path
    X, y = load_dataset(DATA_PATH)

    print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features.")

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                          use_label_encoder=False, eval_metric='mlogloss')

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y_encoded, cv=kf, scoring='accuracy')

    print("\n📊 Cross-Validation Accuracy per Fold:")
    for i, score in enumerate(scores):
        print(f" Fold {i+1}: {score*100:.2f}%")

    print(f"\n✅ Average Accuracy: {np.mean(scores)*100:.2f}%")

    print("\n📌 Training final model and showing classification report...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\n🔬 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

Loaded 2604 samples with 20808 features.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:43:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:48:09] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:52:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:57:41] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:02:30] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e


📊 Cross-Validation Accuracy per Fold:
 Fold 1: 98.46%
 Fold 2: 96.93%
 Fold 3: 96.93%
 Fold 4: 97.50%
 Fold 5: 97.88%

✅ Average Accuracy: 97.54%

📌 Training final model and showing classification report...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:07:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



🔬 Classification Report:
              precision    recall  f1-score   support

      lung_n       0.99      0.87      0.93       101
    lung_scc       0.97      1.00      0.98       420

    accuracy                           0.97       521
   macro avg       0.98      0.93      0.95       521
weighted avg       0.97      0.97      0.97       521



In [ ]:
import cv2
import numpy as np
from skimage.feature import hog, daisy
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import joblib  # Optional if saving/loading model later

# -----------------------------
# HOG + DAISY Feature Extraction
# -----------------------------
def extract_features_from_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")

    img = cv2.resize(img, (150, 150))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys',
                       transform_sqrt=True, feature_vector=True)

    daisy_desc = daisy(gray, step=8, radius=15, rings=2, histograms=6,
                       orientations=8, visualize=False)
    daisy_features = daisy_desc.flatten()

    min_len = min(len(hog_features), len(daisy_features))
    combined = np.hstack([hog_features[:min_len], daisy_features[:min_len]])
    return combined.reshape(1, -1)

# -----------------------------
# Predict Single Image
# -----------------------------
def predict_image_class(image_path, model, label_encoder):
    features = extract_features_from_image(image_path)
    prediction_encoded = model.predict(features)[0]
    predicted_class = label_encoder.inverse_transform([prediction_encoded])[0]
    print(f"🧠 Predicted Class: {predicted_class}")
    return predicted_class

# -----------------------------
# Example: Use Already-Trained Model and Encoder
# -----------------------------
# If model & encoder are in memory from your training code:
#   - Replace these with your trained model & LabelEncoder
# Otherwise, load them with joblib if saved previously.

# If you haven't saved model, assume you just finished training:
# model = <your trained XGBClassifier>
# le = <your trained LabelEncoder>

# Example use
# image_path = 'path_to_your_image.jpg'
# predict_image_class(image_path, model, le)


In [ ]:
image_path = '/content/drive/MyDrive/Minor_Project_image_dataset/Final_Resized/lung_scc/lungscc1066.jpeg'  # Change to your image path
predict_image_class(image_path, model, le)

🧠 Predicted Class: lung_scc


np.str_('lung_scc')